# 10.16 — Consistency & flow-matching models
Consistency and flow-matching models learn a direct path from noise to data, reducing generation to fewer, larger steps. They connect diffusion-style sampling to velocity fields and consistency constraints. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, make_blobs, make_moons
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import StandardScaler

SEED = 1014
rng = np.random.default_rng(SEED)


def normalize01(x):
    arr = np.asarray(x, dtype=float)
    lo = float(np.min(arr))
    hi = float(np.max(arr))
    span = hi - lo
    if span < 1e-12:
        return np.zeros_like(arr)
    return (arr - lo) / span


def synthetic_faces(n=64, size=16, seed=SEED):
    local_rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[-1:1:complex(size), -1:1:complex(size)]
    faces = []
    for i in range(n):
        face = 0.25 + 0.25 * np.exp(-2.0 * (xx ** 2 + yy ** 2))
        eye_y = -0.30 + local_rng.normal(0.0, 0.03)
        eye_dx = 0.35 + local_rng.normal(0.0, 0.02)
        mouth_y = 0.45 + local_rng.normal(0.0, 0.04)
        face -= 0.35 * np.exp(-90.0 * ((xx - eye_dx) ** 2 + (yy - eye_y) ** 2))
        face -= 0.35 * np.exp(-90.0 * ((xx + eye_dx) ** 2 + (yy - eye_y) ** 2))
        face += 0.14 * np.exp(-70.0 * (xx ** 2 + (yy - 0.06) ** 2))
        face -= 0.20 * np.exp(-55.0 * (xx ** 2 + (yy - mouth_y) ** 2))
        face += local_rng.normal(0.0, 0.025, size=(size, size))
        faces.append(normalize01(face))
    return np.asarray(faces).reshape(n, size * size)


def load_tiny_faces():
    try:
        from sklearn.datasets import fetch_olivetti_faces
        faces = fetch_olivetti_faces(download_if_missing=False, shuffle=True, random_state=SEED)
        data = faces.data[:64]
        return data, (64, 64), "Olivetti faces cache"
    except Exception:
        return synthetic_faces(), (16, 16), "synthetic face fallback"


def make_f9_ladder(seed=SEED):
    local_rng = np.random.default_rng(seed)
    d1 = local_rng.normal(0.0, 1.0, size=(80, 1))
    d2, _ = make_moons(n_samples=120, noise=0.08, random_state=seed)
    d3, _ = make_blobs(n_samples=144, centers=4, n_features=6, cluster_std=0.65, random_state=seed)
    digits = load_digits()
    d4 = digits.data[:80] / 16.0
    faces, face_shape, face_source = load_tiny_faces()
    ladder = [
        {"name": "D1 gaussian", "data": d1, "shape": (1, 1), "kind": "vector"},
        {"name": "D2 two moons", "data": d2, "shape": (1, 2), "kind": "points"},
        {"name": "D3 mixture", "data": d3, "shape": (2, 3), "kind": "vector"},
        {"name": "D4 sklearn digits", "data": d4, "shape": (8, 8), "kind": "image"},
        {"name": "D5 faces (" + face_source + ")", "data": faces, "shape": face_shape, "kind": "image"},
    ]
    return ladder


def centered_scaled(data):
    scaler = StandardScaler()
    scaled = scaler.fit_transform(np.asarray(data, dtype=float))
    return scaled, scaler


def choose_components(data, cap=8):
    n_samples, n_features = data.shape
    return max(1, min(cap, n_samples - 1, n_features))


def pca_reconstruct(data, n_components):
    n_components = max(1, min(n_components, data.shape[0] - 1, data.shape[1]))
    pca = PCA(n_components=n_components, random_state=SEED)
    z = pca.fit_transform(data)
    recon = pca.inverse_transform(z)
    return recon, z, pca


def mse(a, b):
    return float(np.mean((np.asarray(a) - np.asarray(b)) ** 2))


def two_sample_distance(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    n = min(len(a), len(b), 80)
    a = a[:n]
    b = b[:n]
    aa = pairwise_distances(a, a).mean()
    bb = pairwise_distances(b, b).mean()
    ab = pairwise_distances(a, b).mean()
    return float(2.0 * ab - aa - bb)


def preview_array(ax, sample, shape, title):
    arr = np.asarray(sample)
    if int(np.prod(shape)) == arr.size and shape[0] > 1 and shape[1] > 1:
        ax.imshow(arr.reshape(shape), cmap="gray")
        ax.set_xticks([])
        ax.set_yticks([])
    else:
        ax.plot(arr.ravel(), marker="o")
    ax.set_title(title, fontsize=9)


def show_ladder_preview(ladder):
    fig, axes = plt.subplots(1, len(ladder), figsize=(14, 2.8))
    for ax, rung in zip(axes, ladder):
        preview_array(ax, rung["data"][0], rung["shape"], rung["name"])
    plt.tight_layout()
    return fig


## The concept, built once (D1)
For a linear path $x_t=(1-t)x_0+t x_1$, flow matching targets $v=x_1-x_0$. With $x_0=0$, $x_1=2$, and $t=0.3$, the lesson gives $x_t=0.6000$, $v=2.000$, and one $0.1$ step gives $0.8000$.

In [ ]:

def flow_matching_path_and_step(x0, x1, t, step_size):
    xt = (1.0 - t) * x0 + t * x1
    velocity = x1 - x0
    next_x = xt + step_size * velocity
    return xt, velocity, next_x

xt, velocity, next_x = flow_matching_path_and_step(0.0, 2.0, 0.3, 0.1)
assert round(float(xt), 4) == 0.6000
assert round(float(velocity), 3) == 2.000
assert round(float(next_x), 4) == 0.8000
print("x_t", round(float(xt), 4), "v", round(float(velocity), 3), "next", round(float(next_x), 4))


Across D1–D5 we simulate a learned transport by mapping Gaussian noise to the target mean and covariance. Sampling quality is measured with a two-sample distance rather than by executing a heavy generative model.

In [ ]:

def covariance_transport_samples(data, seed=SEED, steps=8, path_power=1.0):
    local_rng = np.random.default_rng(seed)
    x = np.asarray(data, dtype=float)
    n, dim = x.shape
    noise = local_rng.normal(0.0, 1.0, size=(n, dim))
    mean = x.mean(axis=0, keepdims=True)
    cov = np.cov(x, rowvar=False) + 1e-4 * np.eye(dim)
    values, vectors = np.linalg.eigh(cov)
    coloring = vectors @ np.diag(np.sqrt(np.maximum(values, 1e-6))) @ vectors.T
    target = mean + noise @ coloring.T
    current = noise.copy()
    for step in range(steps):
        t = (step + 1) / steps
        weight = t ** path_power
        velocity = target - current
        current = current + weight * velocity / steps
    metric = two_sample_distance(x, current)
    return current, metric


## The dataset ladder (D1–D5)
We use the same F9 ladder inline for every topic: a one-dimensional Gaussian, two moons, a mixture, tiny sklearn digits, and a face rung. The face rung uses a local Olivetti cache when present and otherwise a deterministic no-download synthetic face fallback.

In [ ]:

ladder = make_f9_ladder()
for index, rung in enumerate(ladder, start=1):
    data = rung["data"]
    print(f"D{index}: {rung['name']} | shape={data.shape} | sample_shape={rung['shape']} | kind={rung['kind']}")
show_ladder_preview(ladder)


## Run the SAME method across D1-D5
The same reusable method is applied to every rung and one plan metric is collected per rung.

In [ ]:

results = []
for level, rung in enumerate(ladder, start=1):
    samples, metric = covariance_transport_samples(rung["data"], steps=10, path_power=1.0)
    results.append({"level": level, "name": rung["name"], "metric": metric, "sample": samples[0], "shape": rung["shape"]})
    print(f"D{level} {rung['name']}: 2-sample distance={metric:.5f}")


## Results visualization
The closing figure has generated-sample panels for D1-D5 plus the requested metric curve.

In [ ]:

fig, axes = plt.subplots(1, len(results), figsize=(14, 2.8))
for ax, result in zip(axes, results):
    preview_array(ax, result["sample"], result["shape"], result["name"])
plt.suptitle("Generated or reconstructed samples by rung")
plt.tight_layout()

plt.figure(figsize=(6, 3.2))
plt.plot([r["level"] for r in results], [r["metric"] for r in results], marker="o")
plt.xticks([r["level"] for r in results], [r["name"].split()[0] for r in results])
plt.ylabel("2-sample distance")
plt.xlabel("F9 rung")
plt.title("Flow matching transport: metric vs. complexity")
plt.grid(True, alpha=0.3)
plt.tight_layout()


## Pitfall on D5: poor path or too-large one-step sampling
A one-step sampler is not free: large steps amplify approximation error. We compare an overshooting one-step path with smoother multi-step integration on D5.

In [ ]:

d5 = ladder[-1]
data = d5["data"]
bad_samples, bad_metric = covariance_transport_samples(data, steps=1, path_power=2.5)
good_samples, good_metric = covariance_transport_samples(data, steps=16, path_power=1.0)
print(f"one-step distorted 2-sample distance={bad_metric:.5f}")
print(f"multi-step smoother 2-sample distance={good_metric:.5f}")
fig, axes = plt.subplots(1, 3, figsize=(7, 2.4))
preview_array(axes[0], data[0], d5["shape"], "target D5")
preview_array(axes[1], bad_samples[0], d5["shape"], "one step")
preview_array(axes[2], good_samples[0], d5["shape"], "multi step")
plt.tight_layout()


## Evaluate it + Practice
- Track `2-sample distance` against a no-skill baseline such as random samples, unconditioned reconstruction, or independent frames.
- Run a cheap sanity check: dimensions match, finite metrics, and generated samples stay in the training range.
- Ablation: use one oversized Euler step or a nonlinear path that reaches the data too abruptly; the metric should get worse.
- Failure signals: D5 looks plausible in one panel but the metric curve jumps, indicating artifacts hidden by small previews.

Practice 1: change the latent or conditioning dimension and predict which rung changes most.


Practice 2: change `path_power` and find when D2 moons distort first.

Practice 3: compare two-sample distance with plain reconstruction MSE and explain the difference.